# Burfisher 3e: ME9A

Replicates exercise ME9A of Burfisher, *Introduction to Computable General Equilibrium Models* (3rd ed.), from scratch, on the book's nus333 database (2 regions × 3 sectors), and every level is compared with GEMPACK's solution of the same `.EXP`.

Run it from anywhere inside the repository.

In [ ]:
import sys
from pathlib import Path

# the repository root, wherever the notebook is opened from
ROOT = next(d for d in (Path.cwd(), *Path.cwd().parents) if (d / "pyproject.toml").exists())
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "scripts" / "gtap")]

from pyomo.environ import value

from equilibria._local_refs import nus333_dir, path_capi_src
from equilibria.blocks.gtap import ClosureBlock, ShockBlock, overwrite, ppriv_tornqvist
from equilibria.blocks.gtap._derived_params import xscale_data
from equilibria.templates.gtap import GTAPParameters
from equilibria.templates.gtap.gtap_block_model import build_block_model
from equilibria.templates.gtap.gtap_contract import GTAPClosureConfig
from equilibria.templates.gtap.gtap_multiperiod_driver import solve_multiperiod
from equilibria.templates.gtap.shocks import apply_shock

sys.path.insert(0, str(path_capi_src()))  # the PATH solver's Python bindings

## 1. Data: nus333 and the exercise's parameters (`climatechange.prm`)

In [ ]:
har = nus333_dir()
p = GTAPParameters()
p.load_from_har(
    basedata_path=har / "basedata.har",
    sets_path=har / "sets.har",
    default_path=har / "climatechange.prm",
    baserate_path=har / "baserate.har",
)

## 2. The exercise's closure, with `@overwrite`

A 2010-2050 baseline: real GDP grows by a given amount and productivity adjusts. GEMPACK's `ME9A.EXP` swaps `qgdp(REG)` for `aoreg(REG)`. Here:

- productivity `axp` becomes endogenous in every activity, and `b.target` fixes real GDP `rgdpmp[shock] = gdp_target × rgdpmp[check]`;
- `eq_aoreg` keeps the productivity change the same in the three activities of a region, as `aoreg` does.

**Not yet within 0.5% of GEMPACK:** up to 0.77% (`pe` ROW LAND), while it matches GAMS. GEMPACK's own log rates many of its ME9A results to 2 figures; a rerun with more steps will decide which side moves.

In [ ]:
REGS = ("USA", "ROW")
ACTS = ("AGR", "MFG", "SER")


@overwrite(ShockBlock, period="shock")
def baseline(b):
    for r in REGS:
        for a in ACTS:
            b.endogenous("axp", (r, a))
        b.target("gdp_target", (r,), quantity=lambda m, r: m.rgdpmp[r], domains=("r",))


@overwrite(ClosureBlock, period="shock")
def uniform_productivity(b):
    for r in REGS:
        for a in ("MFG", "SER"):
            b.equation(
                "eq_aoreg",
                (r, a),
                lambda m, r, a: m.axp[r, a] / value(m.axp[r, a])
                == m.axp[r, "AGR"] / value(m.axp[r, "AGR"]),
                domains=("r", "a"),
            )

## 3. GTAP closure and model (base / check / shock)

In [ ]:
gc = GTAPClosureConfig(
    name="base",
    closure_type="MCP",
    capital_mobility="sluggish",
    fix_endowments=False,
    fix_taxes=False,
    fix_technology=False,
    if_sub=False,
    savf_flag="capFlex",
    numeraire="pnum",
)
# Like GAMS compStat, the base is not solved: its levels are the reference of the
# Fisher indices of check and shock.
m, _ = build_block_model(p, p.sets, gc, "ROW", base_calibrated=False, ref_gdx=None)

## 4. The shock

As in the `.EXP`, in % change; `apply_shock` converts each instrument to levels and writes it in the shock period only.

In [ ]:
SHOCK = {
    "gdp_target": {
        ("USA",): 109.6,
        ("ROW",): 284.5,
    },
    "pop": {
        ("USA",): 32.3,
        ("ROW",): 37.3,
    },
    "aft": {
        ("USA", "LABOR"): 24.1,
        ("ROW", "LABOR"): 38.4,
        ("USA", "CAPITAL"): 60.6,
        ("ROW", "CAPITAL"): 213.1,
        ("USA", "LAND"): -0.93,
        ("ROW", "LAND"): 4.4,
    },
}
apply_shock(m, SHOCK)

## 5. Solve check and shock

In [ ]:
res = solve_multiperiod(
    m,
    p,
    gc,
    ref_gdx=None,
    skip_base_solve=True,
    mute_welfare=True,
    seed_from_prior=False,
    mode="gtap",
    solve_check=True,
    # the shock in 4 steps, each solve starting from the previous one
    shock_steps=(0.25, 0.5, 0.75, 1.0),
)
codes = {k: int(res[k]["code"]) for k in res}
print("codes:", codes)
assert set(codes.values()) == {1}, codes

## 6. Every level against GEMPACK

In [ ]:
import compare_burfisher_gempack as cbg
import compare_gempack_levels as cg

EXP = "ME9A"
GEMPACK_DIR = ROOT / "tests" / "fixtures" / "gtap7_gempack" / "burfisher_gragg"
report = cbg.run(EXP, cg.model_levels(m), dict(xscale_data(p, p.sets)), GEMPACK_DIR)
o = report.overall
w = o["pct_within"]
print(f"{o['cells']} cells (pre and post) against GEMPACK {cbg.method(EXP, GEMPACK_DIR)}")
print(f"within 0.1%: {w[1e-3]:.2f}%   within 0.5%: {w[5e-3]:.2f}%   within 1%: {w[1e-2]:.2f}%")
print("\nlargest differences:")
print(f"{'variable':10s}{'period':>7s}  {'cell':24s}{'equilibria':>14s}{'GEMPACK':>14s}{'diff':>9s}")
for c in sorted(report.cells, key=lambda c: -c.d)[:5]:
    cell = ",".join(c.index)
    print(f"{c.variable:10s}{c.period:>7s}  {cell:24s}{c.a:14.6g}{c.b:14.6g}{100 * c.d:8.3f}%")